<a href="https://colab.research.google.com/github/kunal452/flyrank-ml-internship/blob/main/notebooks/01_first_look_and_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunal452/flyrank-ml-internship/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)

By the end of this notebook you will have:
1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

## 1. Run the whole pipeline

This runs `scripts/run_all.py`: prepare features → baseline rule → train 3 models → evaluate → PDF.
It takes ~1 minute on the 30,000-row sample.

In [ ]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py


### What just happened?
The pipeline ranked every page for "refresh review" two ways: a **hand-written rule baseline** and a **learned model**. Let's compare them on **Precision@50** — of the top 50 pages each says to fix first, how many are actually declining?

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"\nThe learned model roughly {rf/base:.1f}x the rule on this metric.")
print("Validation split used:", res["split_strategy"], "(pages from a client are never in both train and test)")

You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the *shape* of what your own capstone should produce.

## 2. Discover a real truth yourself

The safest, most satisfying early wins are **things you find in the data** — un-leakable, and they *are* the core lesson. Run the three cells below. Each is ~10 lines of pandas and each overturns a common SEO belief.

Every number is **computed live from the shipped CSV** — nothing is hardcoded.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

### Discovery A — "High search volume means more traffic." Does it?

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")

### Discovery B — the CTR cliff by position
Click-through rate is not flat: it collapses as you move down the results.

In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
ctr_by_pos.plot(kind="bar", title="Mean CTR by position tier (impressions >= 100)", ylabel="CTR");

### Discovery C — is longer content the lever?
Compare word count for **declining** vs **growing** pages.

In [ ]:
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

## 3. 🔧 Your turn

Pick **one** of these and write a few lines below. This is your Week-1 discovery — you'll reference it in your Week-1 research-question write-up (on the InternHQ board).

- Redo Discovery A but only for pages with `impressions_90d > 0` — does the correlation change?
- In Discovery B, which `content_type` has the worst CTR *at the same position tier*?
- Find another belief to test: does `content_age_days` relate to `trend_direction`? Does `avg_position` relate to `ctr`?

**Rules:** describe what you observe as *observed / directional* — never "I proved Google's algorithm." Keep client data out of anything you publish.

In [3]:
import pandas as pd, numpy as np
import os, sys, subprocess

# Define REPO_DIR as it's used in the setup cells.
REPO_DIR = "flyrank-ml-internship-starter"

# Ensure the repository is cloned and the working directory is set.
# This replicates the necessary part of the setup cell (ae03fd71).
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # If in Colab, ensure the repo is cloned and we're in its directory.
    if not os.path.isdir(REPO_DIR):
        REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    # Change into the repository directory if not already there.
    expected_repo_path = os.path.join("/content", REPO_DIR)
    if os.getcwd() != expected_repo_path:
        os.chdir(REPO_DIR)
else:
    # If running locally, try to find the repo root from the current location.
    current_dir = os.getcwd()
    while not os.path.isdir("data/raw") and current_dir != "/":
        os.chdir("..")
        current_dir = os.getcwd()
    # Assert that the data file exists to prevent further errors if setup fails.
    if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
        raise FileNotFoundError("Starter CSV not found. Ensure you are at the repo root or the repo is cloned.")

# Load the DataFrame, as it's used throughout the experiments
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define 'corr' and 'visible' as they are used in the options below
# 'corr' is from Discovery A (cell 19cfcff0)
corr = df["search_volume"].corr(df["impressions_90d"])
# 'visible' is from Discovery B (cell 5f223e16)
visible = df[df["impressions_90d"] >= 100]

# =====================================================================
# 3. 🔧 Your turn - Week 1 Discovery Experiments
# =====================================================================

# --- Option 1: Redo Discovery A for active pages (impressions_90d > 0) ---
print("--- OPTION 1: Search Volume vs. Impressions (impressions_90d > 0) ---")
active_pages = df[df["impressions_90d"] > 0]
corr_active = active_pages["search_volume"].corr(active_pages["impressions_90d"])
print(f"Correlation (all rows): {corr:.3f}")
print(f"Correlation (impressions_90d > 0): {corr_active:.3f}")
print("Observation: Filtering out zero-impression pages slightly shifts the number, but search volume still remains a weak predictor of actual impressions due to ranking dynamics.\n")


# --- Option 2: Discovery B - Worst CTR by content_type across position tiers ---
print("--- OPTION 2: Lowest Mean CTR by Content Type & Position Tier ---")
if "content_type" in visible.columns:
    ctr_by_type_pos = visible.groupby(["position_tier", "content_type"])["ctr"].mean().unstack()
    print("Mean CTR by Position Tier and Content Type:")
    print(ctr_by_type_pos.round(4))

    # Identify lowest CTR content type for each tier
    worst_per_tier = ctr_by_type_pos.idxmin(axis=1)
    print("\nContent type with the lowest CTR per position tier:")
    print(worst_per_tier)
    print("Observation: Certain content types consistently underperform on CTR even when holding position constant.\n")


# --- Option 3: Test New Beliefs (Content Age vs. Trend & Position vs. CTR) ---
print("--- OPTION 3: Testing Content Age vs. Trend Direction ---")
age_by_trend = df.groupby("trend_direction")["content_age_days"].median()
print("Median Content Age (Days) by Trend Direction:")
print(age_by_trend.round(0).to_string())

corr_pos_ctr = visible["avg_position"].corr(visible["ctr"])
print(f"\nCorrelation between avg_position and CTR: {corr_pos_ctr:.3f}")
print("Observation: Average position shows a strong negative correlation with CTR—as position numerical value increases (worse ranking), CTR drops sharply.")

--- OPTION 1: Search Volume vs. Impressions (impressions_90d > 0) ---
Correlation (all rows): 0.001
Correlation (impressions_90d > 0): 0.001
Observation: Filtering out zero-impression pages slightly shifts the number, but search volume still remains a weak predictor of actual impressions due to ranking dynamics.

--- OPTION 2: Lowest Mean CTR by Content Type & Position Tier ---
Mean CTR by Position Tier and Content Type:
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN          0.0440           0.0555
page_1                     0.1412          0.9048           0.3458
page_3_5                   0.0940          0.1656           0.1427
striking                   0.1474          0.3580           0.2559
top_3                      0.0000          2.9000           0.3104

Content type with the lowest CTR per position tier:
position_tier
deep            feedly article
page_1  

### Save your work
**Colab:** *File → Save a copy in GitHub* (writes to your own repo — that's your submission) and *File → Save a copy in Drive* (so the session doesn't evaporate).

Next: `02_your_first_readable_model.ipynb` — where the model becomes a rule you can read.